# NB-601: Building RAG Pipeline

Build a complete Retrieval-Augmented Generation system.

## Learning Objectives
- Create document embeddings
- Build vector store
- Implement retrieval
- Generate responses with context

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

print("Libraries imported")

## 1. Create Document Store

In [ ]:
# TODO: Sample documents
documents = [
    "Paris is the capital of France and known for the Eiffel Tower.",
    "London is the capital of England and home to Big Ben.",
    "Tokyo is the capital of Japan known for its technology.",
    "New York City is in the United States.",
    "Berlin is the capital of Germany.",
    "Rome is the capital of Italy.",
]

print(f"Documents: {len(documents)}")

## 2. Create Embeddings

In [ ]:
# TODO: Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings
doc_embeddings = embedder.encode(documents)

print(f"Embedding shape: {doc_embeddings.shape}")
print(f"Embedding dim: {doc_embeddings.shape[1]}")

## 3. Build Vector Store

In [ ]:
class SimpleVectorStore:
    def __init__(self, embeddings, documents):
        self.embeddings = np.array(embeddings)
        self.documents = documents

    def search(self, query_embedding, top_k=3):
        # TODO: Compute similarities
        similarities = np.dot(self.embeddings, query_embedding)

        # Get top-k
        top_indices = similarities.argsort()[-top_k:][::-1]

        results = [
            (self.documents[i], similarities[i])
            for i in top_indices
        ]

        return results

# Create vector store
vector_store = SimpleVectorStore(doc_embeddings, documents)
print("Vector store created")

## 4. Implement Retrieval

In [ ]:
# TODO: Retrieve relevant documents
query = "What is the capital of France?"
query_embedding = embedder.encode([query])[0]

retrieved_docs = vector_store.search(query_embedding, top_k=2)

print(f"Query: {query}\n")
for doc, score in retrieved_docs:
    print(f"[{score:.3f}] {doc}")

## 5. Build RAG Pipeline

In [ ]:
class RAGPipeline:
    def __init__(self, vector_store, embedder, model, tokenizer):
        self.vector_store = vector_store
        self.embedder = embedder
        self.model = model
        self.tokenizer = tokenizer

    def query(self, user_query, top_k=2, max_new_tokens=50):
        # TODO: Retrieve relevant documents
        query_embedding = self.embedder.encode([user_query])[0]
        retrieved = self.vector_store.search(query_embedding, top_k=top_k)

        # TODO: Build prompt with context
        context = "\n".join([doc for doc, _ in retrieved])

        prompt = f"""Context:
{context}

Question: {user_query}

Answer:"""

        # TODO: Generate response
        inputs = self.tokenizer(prompt, return_tensors="pt")

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        return {
            "response": response,
            "retrieved_docs": retrieved,
        }

# TODO: Initialize pipeline
# model = AutoModelForCausalLM.from_pretrained("gpt2")
# tokenizer = AutoTokenizer.from_pretrained("gpt2")

# rag = RAGPipeline(vector_store, embedder, model, tokenizer)

print("RAG Pipeline class defined")

## 6. Test RAG System

In [ ]:
# TODO: Query the RAG system
# result = rag.query("Tell me about Paris")

# print(f"Response: {result['response']}\n")
# print("Retrieved documents:")
# for doc, score in result['retrieved_docs']:
#     print(f"  [{score:.3f}] {doc}")

print("RAG system ready to test")

## Exercise

1. Add more documents to the store
2. Implement re-ranking of retrieved documents
3. Add source citations to responses